In [ ]:
# read the data
import matplotlib.pyplot as plt
import numpy as np
#import pandas as pd 
import matplotlib
import sys     
import os
import glob
import mne
from mne.preprocessing import ICA
from scipy import stats
import seaborn as sns
import pandas as pd
#%matplotlib qt
sys.path.append(os.path.abspath('..'))
from utils import identify_bad_channels,remove_epochs_with_bad_mmn_channels, Average_in_trials_in_time_window,remove_or_interpolate_epochs_with_n_bad_chn
from stats_utils import extract_mmn_latencies,align_and_standardize_evokeds,extract_peak_amplitude,extract_mean_amplitudes,snap_times_to_evoked
from visualization import plot_erps
sys.path.append(os.path.abspath('..'))
from notifmmn import erputils

from scipy.stats import ttest_ind, ttest_rel
from statsmodels.stats.multitest import multipletests


# collect subject 

In [ ]:
SN_devi_eps_evoked_dict  = {}
BN_std_eps_evoked_dict  = {}
BN_devi_eps_evoked_dict = {}
SN_std_eps_evoked_dict = {}
time_len ={}
epoch_folder = r"E:\ecf-exp2-notif-mmn\data\Processed\preprocessed_epoched"
for sub in os.listdir(epoch_folder ):
    if (sub=="CAR"):
        continue
    print(sub)
    epochs = mne.read_epochs(os.path.join(epoch_folder,sub), preload=True)
    print("BN STD : ", len(epochs['S14']),"SN STD : ", len(epochs['S24']) )
    # sn_N1_time = SN_N1_SME[sub[-13:-8]]['PL_mean']-SN_N1_SME[sub[-13:-8]]['PL_std']
    # bn_N1_time = BN_N1_SME[sub[-13:-8]]['PL_mean']-BN_N1_SME[sub[-13:-8]]['PL_std']  
    sn_N1_time= score_df[(score_df['sub_id']==sub[:-8])& (score_df['measure']=='n1')]['SN_PL_mean'].iloc[0]
    bn_N1_time = score_df[(score_df['sub_id']==sub[:-8])& (score_df['measure']=='n1')]['BN_PL_mean'].iloc[0]
    # sn_P2_time = SN_P2_SME[sub[-13:-8]]['PL_mean']+SN_P2_SME[sub[-13:-8]]['PL_std']
    # bn_P2_time = BN_P2_SME[sub[-13:-8]]['PL_mean']+BN_P2_SME[sub[-13:-8]]['PL_std']
    # sn_time_win = (0, sn_P2_time - sn_N1_time)  
    # bn_time_win = (0, bn_P2_time - bn_N1_time)
    print(sn_N1_time,bn_N1_time)
    epochs_times = epochs.times
    # find nearest index
    nearest_bn_N1_time= epochs_times[np.argmin(np.abs(epochs_times - bn_N1_time))]
    nearest_sn_N1_time= epochs_times[np.argmin(np.abs(epochs_times - sn_N1_time))]

    sn_epochs = epochs['S15','S24'].copy().shift_time(-nearest_sn_N1_time)  # shift time backward by 100 ms
    bn_epochs = epochs['S14','S25'].copy().shift_time(-nearest_bn_N1_time)
    
    SN_devi_eps = sn_epochs['S15'].average()
    BN_std_eps = bn_epochs['S14'].average()
    BN_devi_eps = bn_epochs['S25'].average()
    SN_std_eps = sn_epochs['S24'].average() 
    tmin = max(SN_devi_eps.times[0],BN_std_eps.times[0],
               BN_devi_eps.times[0],SN_std_eps.times[0])
    tmax = min(SN_devi_eps.times[-1],BN_std_eps.times[-1],
               BN_devi_eps.times[-1],SN_std_eps.times[-1],)
    SN_devi_eps_evoked_dict[sub[:-8]] = SN_devi_eps.copy().crop(tmin, tmax)
    BN_devi_eps_evoked_dict[sub[:-8]] = BN_devi_eps.copy().crop(tmin, tmax)
    SN_std_eps_evoked_dict[sub[:-8]] = SN_std_eps.copy().crop(tmin, tmax)
    BN_std_eps_evoked_dict[sub[:-8]] = BN_std_eps.copy().crop(tmin, tmax)
    time_len[sub[:-8]] = (tmin,tmax)
   

# Group LSU and HSU define

# Filter STD N1 and P2 subject  

# Deviant N1 